# 分類結果を評価する (適合率・再現率・F1・混同行列)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/09_classification/classification_report.ipynb)

NLP クックブック 09-02 ／ テキスト分類

正解率だけでなく、クラスごとの適合率・再現率・F1 と、どのクラスをどのクラスに間違えたかを見たい。

In [ ]:
!pip install -q matplotlib-fontja

In [ ]:
import io, tarfile, urllib.request
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline

def load_livedoor():
    url = "https://www.rondhuit.com/download/ldcc-20140209.tar.gz"
    with urllib.request.urlopen(url) as r:
        tar = tarfile.open(fileobj=io.BytesIO(r.read()), mode="r:gz")
    rows = []
    for m in tar.getmembers():
        parts = m.name.split("/")
        if m.isfile() and len(parts) == 3 and parts[2] != "LICENSE.txt":
            lines = tar.extractfile(m).read().decode("utf-8").split("\n")
            rows.append({"label": parts[1], "title": lines[2], "text": "\n".join(lines[3:])})
    return pd.DataFrame(rows)

df = load_livedoor()
train, test = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=0)
# タイトルだけを使う難しめの設定。文字 n-gram なら分かち書きも不要
clf = make_pipeline(TfidfVectorizer(analyzer="char", ngram_range=(1, 3), sublinear_tf=True), LogisticRegression(C=10, max_iter=1000))
clf.fit(train["title"], train["label"])
pred = clf.predict(test["title"])

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, classification_report, f1_score

print(classification_report(test["label"], pred, digits=3))
print("macro-F1:", f1_score(test["label"], pred, average="macro").round(3),
      " micro-F1 (= 正解率):", f1_score(test["label"], pred, average="micro").round(3))

dummy = DummyClassifier(strategy="most_frequent").fit(train["title"], train["label"])
print("最頻クラスを答えるだけのベースライン:", accuracy_score(test["label"], dummy.predict(test["title"])).round(3))

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay.from_predictions(test["label"], pred, normalize="true", values_format=".2f",
                                        xticks_rotation=60, cmap="Blues", ax=ax, colorbar=False)
plt.show()

- macro-F1 はクラスごとの F1 の単純平均で、少数クラスも同じ重みで評価する。不均衡データでは正解率より macro-F1 を見る。
- 混同行列は行が正解、列が予測。`normalize="true"` で行ごとの割合 (= 再現率) になる。
- 必ず「何も考えないベースライン」と比べる。